# Healthcare Liver Disease Decision Support — Milestone 1
**Participant:** Participant_M1_02  
**Project:** Healthcare Liver Disease Decision Support  
**Milestone:** 1  
**Dataset package:** `HealthCareData.zip`

## Objective
Predict liver disease status for educational decision support, explain model factors, and communicate risk responsibly.

## Milestone scope
join patient data and status using patient ID; inspect quality and target codes; EDA; preprocess; baseline Logistic Regression; evaluate precision, recall, F1, ROC-AUC.

> This is a synthetic sample submission for evaluator testing. Dataset files must be placed in the paths referenced below. Results are not pre-filled or fabricated.

## 1. Load clinical data and target status
The brief defines `Status`: 1 = Liver Patient and 2 = Non-Liver Patient. Exclude `ID` from model features.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("data")
patients = pd.read_csv(DATA_DIR / "Liver Patient Dataset.csv")
status = pd.read_excel(DATA_DIR / "Status.xlsx")
print(patients.shape, status.shape)
print(patients.dtypes)
display(patients.head())
display(status.head())

In [ ]:
# Inspect keys before joining; adjust the key name only after inspecting source columns.
print("patient columns:", patients.columns.tolist())
print("status columns:", status.columns.tolist())
print("missing:", patients.isna().sum().sort_values(ascending=False).head(12))
print("duplicate rows:", patients.duplicated().sum())

## 2. EDA and data preparation
Investigate class balance and clinical measurements. Extreme values require clinical/data-quality investigation rather than automatic deletion.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

target = "Status"
if target in patients.columns:
    display(patients[target].value_counts(dropna=False))
for col in ["Age", "TB", "DB", "Alkphos", "Sgpt", "Sgot", "TP", "ALB", "A/G Ratio"]:
    if col in patients.columns:
        sns.histplot(patients[col].dropna(), kde=True)
        plt.title(col)
        plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# If target resides in Status.xlsx, join using the verified patient ID column before this step.
df = patients.copy()
if target not in df.columns:
    raise ValueError("Join the Status.xlsx target using the verified patient ID column before modelling.")
y = df[target].map({1: 1, 2: 0})
X = df.drop(columns=[target, "ID"], errors="ignore")
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=np.number).columns.tolist()
prep = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
clf = Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))])
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
prob = clf.predict_proba(X_test)[:, 1]
print(classification_report(y_test, pred, zero_division=0))
print(confusion_matrix(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, prob))

## 3. Responsible interpretation
This is an educational model, not a diagnostic system. Report false negatives, class balance, limitations, and the need for qualified clinical judgement.